In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 58. Week 40 — Experiment infrastructure, registry, drift, and rollback

## 学習目標

- config/data/code/metric/artifactをcontent-addressed runへ固定できる
- append-only run evidenceとproduction pointerを分離できる
- reference-fixed drift diagnosticを計算できる
- batch inferenceのmodel/input/output lineageを保存できる
- alert、promotion、rollbackの事前規則を書ける

## 前提知識

- Week 38のpackage/config contract
- Week 39のavailabilityとschema
- B9 model selection gate

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 58


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask
assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("locked outer rows present: False")

fixture rows: 256
inner train / validation: 192 64
locked outer rows present: False


In [4]:
numeric_preprocessor = qt.fit_numeric_preprocessor(fixture.numeric_features, train_mask)
numeric_features = numeric_preprocessor.transform(fixture.numeric_features)
numeric_model = qt.fit_sparse_ridge(
    numeric_features[train_mask], fixture.targets[train_mask], ridge=1.0
)
numeric_validation_prediction = numeric_model.predict(numeric_features[validation_mask])
numeric_metrics = qt.regression_error_table(
    fixture.targets[validation_mask],
    numeric_validation_prediction,
    np.asarray(fixture.entity_ids)[validation_mask],
)
print("numeric validation metrics:", numeric_metrics)

numeric validation metrics: {'mae': 0.06965156975112882, 'median_absolute_error': 0.02798038529997949, 'rmse': 0.15698922708171414, 'company_macro_mae': 0.05985240327713495}


## 1. Content-addressed runとregistry

run IDはhuman labelでなくcanonical config、data hash、code revision、metric、artifact hashから作る。registryはevidenceを上書きせずappendし、promotion/rollbackはpointerを動かす。教材fixtureのrunをproductionへpromoteしない。

In [5]:
import hashlib

data_digest = "953c9b06c6c1dc1ef68c5e21f1ee88c4fe20d1ee34d5887150e51843184ad0b0"
prediction_digest = hashlib.sha256(numeric_validation_prediction.astype("<f8").tobytes()).hexdigest()
development_run = qt.build_experiment_run(
    experiment_name="b10-infrastructure-lab",
    candidate_name="numeric-ridge",
    stage="development",
    config={"ridge": 1.0, "outer_access": "unopened"},
    data_sha256=data_digest,
    code_revision="notebook-58-generated-source",
    metrics=numeric_metrics,
    artifact_sha256={"validation_prediction": prediction_digest},
)
registry = qt.register_run(qt.ModelRegistry(), development_run)
assert registry.production_run_id is None
print("registered run:", development_run.run_id)
print("production pointer:", registry.production_run_id)

registered run: 5be7538a84cf5fed57c98652
production pointer: None


## 2. Drift is a diagnostic, not a model verdict

reference training quantileを固定してPSIを計算し、KS statisticを併記する。p値はsample size依存であり、PSI 0.1/0.25等の慣用値も普遍定数ではない。threshold、action、minimum sample、seasonal exclusionをmonitoring前に固定する。

In [6]:
feature_index = fixture.numeric_feature_names.index("log_previous_assets")
reference = fixture.numeric_features[train_mask, feature_index]
current = fixture.numeric_features[validation_mask, feature_index]
drift = qt.numeric_drift_report(reference, current, bins=8)

fig = go.Figure()
labels = [f"bin {index}" for index in range(drift.reference_proportions.size)]
fig.add_bar(x=labels, y=drift.reference_proportions, name="inner train")
fig.add_bar(x=labels, y=drift.current_proportions, name="inner validation")
fig.update_layout(title="Reference-fixed feature drift bins", yaxis_title="Proportion", barmode="group", template="plotly_white")
fig.show()
print("PSI:", drift.population_stability_index)
print("KS statistic / p-value:", drift.ks_statistic, drift.ks_pvalue)

PSI: 0.20834707690780904
KS statistic / p-value: 0.21875 0.018484138096300142


In [7]:
feature_input_digest = hashlib.sha256(numeric_features[validation_mask].astype("<f8").tobytes()).hexdigest()
batch_result = qt.batch_inference(
    numeric_model.predict,
    numeric_features[validation_mask],
    model_run_id=development_run.run_id,
    input_sha256=feature_input_digest,
)
np.testing.assert_array_equal(batch_result.predictions, numeric_validation_prediction)
display(pd.DataFrame([{"model_run_id": batch_result.model_run_id, "input_sha256": batch_result.input_sha256, "output_sha256": batch_result.output_sha256, "rows": batch_result.row_count}]))

,model_run_id,input_sha256,output_sha256,rows
0,5be7538a84cf5fed57c98652,3612e45d47c91b0217fabdd967404bb017731945396ace...,0cd72c914c8cd79c2ff190492e4130d965b9ddce54c50a...,64


## 3. Batch/online/rollback boundary

batch inferenceはimmutable input snapshotへ同じmodelを適用しやすい。online inferenceはfeature freshness、concurrency、latency、partial failure、serving/training skewを追加する。rollbackは旧run artifactとschemaが利用可能であることを事前testし、model pointerだけでなくfeature/data compatibilityも確認する。

## 4. 失敗モード

- mutable file pathをmodel versionと呼ぶ
- config変更後も同じrun IDを使う
- drift alert後にthresholdを変更する
- validation driftだけでproduction rollbackする
- online endpointを作っただけでreproducibleと呼ぶ

## 5. 段階別演習

### 基礎

1. run IDに結ぶ5種類のlineageを書け。
2. batch input/output hashを再計算せよ。

### 標準

3. candidate run 2件のpromotion/rollback testを書け。
4. drift thresholdとaction matrixを事前登録せよ。

### 研究

5. training-serving skewを検知するshadow inference計画を書け。

## 6. Exit Criteria

- [ ] run evidenceをappend-onlyにした
- [ ] production pointerとartifactを分離した
- [ ] reference binをcurrent dataで再fitしていない
- [ ] batch model/input/output hashを保存した
- [ ] 教材runをproductionへpromoteしていない

## 7. 出典

- [Python Packaging User Guide](https://packaging.python.org/en/latest/)
- [Python logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [pytest documentation](https://docs.pytest.org/)
- [Semantic Versioning 2.0.0](https://semver.org/)